# California Housing Regression Hyperparameter Tuning

Contains the frozen-backprop sigma search, the documented local 3x3 grid, and an editable full-length run cell.

In [ ]:
from pathlib import Path
import os
import shutil
import sys


def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _looks_like_project_root(path: Path) -> bool:
    return (path / "learning_rules_MLP.py").is_file() and (path / "experiment_utils").is_dir()


def _find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for env_name in ["PROJECT_ROOT", "COLAB_PROJECT_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value))
    candidates.extend([
        Path("/content/backprop-alternatives"),
        Path("/content/drive/MyDrive/backprop-alternatives"),
        Path("/content/drive/MyDrive/colab-folder"),
        Path("/content/drive/MyDrive/Colab Notebooks/drive-folder"),
    ])
    for start in candidates:
        try:
            resolved = start.expanduser().resolve()
        except Exception:
            continue
        for candidate in [resolved, *resolved.parents]:
            if _looks_like_project_root(candidate):
                return candidate
    if _running_in_colab():
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        for hit in Path("/content/drive/MyDrive").rglob("learning_rules_MLP.py"):
            candidate = hit.parent
            if _looks_like_project_root(candidate):
                return candidate
    raise FileNotFoundError("Could not find a folder containing learning_rules_MLP.py and experiment_utils/.")


def _stage_code_locally_if_colab(source_root: Path) -> Path:
    """Import code from /content in Colab instead of reading Python modules from Drive."""
    if not _running_in_colab() or not str(source_root).startswith("/content/drive/"):
        return source_root

    runtime_root = Path("/content/backprop-alternatives-runtime")
    if runtime_root.exists():
        shutil.rmtree(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_root / "learning_rules_MLP.py", runtime_root / "learning_rules_MLP.py")
    shutil.copytree(
        source_root / "experiment_utils",
        runtime_root / "experiment_utils",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"),
    )
    (runtime_root / "notebooks").mkdir(exist_ok=True)
    return runtime_root


SOURCE_PROJECT_ROOT = _find_project_root()
PROJECT_ROOT = _stage_code_locally_if_colab(SOURCE_PROJECT_ROOT)
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from experiment_utils.runtime import get_device, setup_matplotlib

setup_matplotlib()
DEVICE = get_device()
print(f"Source project root: {SOURCE_PROJECT_ROOT}")
print(f"Runtime project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Device: {DEVICE}")


In [ ]:
NOTEBOOK_NAME = "california-housing-hyperparam-tuning"

TASK_CONFIG = {
    "task_key": "california_housing",
    "display_name": "California housing regression",
    "task_type": "regression",
    "data_loader": "load_california_housing",
    "activation": "sigmoid",
    "dimensions": (8, 128, 64, 1),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "seeds": [0],
    "data_kwargs": {
        "test_size": 0.2,
        "batch_size": 256,
        "eval_batch_size": 4096,
        "seed": 0,
        "num_workers": 0,
        "data_home": str(DATA_DIR / "sklearn"),
    },
    "run_epochs": 200,
    "run_print_every_epoch": 25,
    "sigma_search": {
        "epochs": 50,
        "bp_lr": 0.010,
        "num_perturbations": 50,
        "batch_size": 256,
        "checkpoint_epochs": [1, 25, 50],
        "seeds": [0],
        "sigma_grids": {
            "np": [0.060, 0.080, 0.100, 0.120, 0.140],
            "np_fan_in": [0.0375, 0.0450, 0.0525, 0.0600, 0.0675],
            "np_fixed": [0.350, 0.400, 0.450, 0.500, 0.550],
            "wp": [0.060, 0.080, 0.100, 0.120, 0.140],
        },
    },
    "grid_search": {
        "epochs": 30,
        "seeds": [0],
        "print_every_epoch": 10,
        "local_grid": {
            "bp": {"lr": [0.075, 0.100, 0.150]},
            "np": {"lr": [0.0200, 0.0270, 0.0350], "sigma": [0.080, 0.100, 0.120]},
            "np_fan_in": {"lr": [0.0150, 0.0175, 0.0200], "sigma": [0.0450, 0.0525, 0.0600]},
            "np_fixed": {"lr": [0.0150, 0.0175, 0.0200], "sigma": [0.400, 0.450, 0.500]},
            "wp": {"lr": [0.0110, 0.0135, 0.0160], "sigma": [0.080, 0.100, 0.120]},
        },
    },
    "full_run_epochs": 200,
    "full_run_seeds": [0],
}

FULL_RUN_CONFIGS = {
    "bp": {"lr": 0.100},
    "np": {"lr": 0.0270, "sigma": 0.100},
    "np_fan_in": {"lr": 0.0175, "sigma": 0.0525},
    "np_fixed": {"lr": 0.0175, "sigma": 0.450},
    "wp": {"lr": 0.0135, "sigma": 0.100},
}


In [ ]:
from experiment_utils.sigma_search import run_sigma_search

sigma_outputs = run_sigma_search(TASK_CONFIG, project_root=PROJECT_ROOT, show=True, device=DEVICE)


In [ ]:
from experiment_utils.grid_search import best_grid_rows, run_local_grid_search
from IPython.display import display

grid_outputs = run_local_grid_search(TASK_CONFIG, project_root=PROJECT_ROOT, show=True, device=DEVICE)
print("Best row per method")
display(best_grid_rows(grid_outputs["grid_summary_df"]))


In [ ]:
# Editable full-length run. Change FULL_RUN_CONFIGS above, then set RUN_FULL_LENGTH=True.
RUN_FULL_LENGTH = False

if RUN_FULL_LENGTH:
    from experiment_utils.grid_search import run_full_length_training

    full_outputs = run_full_length_training(
        TASK_CONFIG,
        run_configs=FULL_RUN_CONFIGS,
        project_root=PROJECT_ROOT,
        show=True,
        device=DEVICE,
    )


In [ ]:
# Optional manual export/download. Set EXPORT_RESULTS=True after the runs have completed.
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    from experiment_utils.export import download_if_colab, export_outputs

    export_bundle = {}
    for prefix, obj_name in [
        ("sigma", "sigma_outputs"),
        ("grid", "grid_outputs"),
        ("full", "full_outputs"),
    ]:
        if obj_name in globals():
            for key, value in globals()[obj_name].items():
                export_bundle[f"{prefix}_{key}"] = value
    archive_path = export_outputs(export_bundle, archive_name=NOTEBOOK_NAME)
    download_if_colab(archive_path)
